# 02 - Limpeza de dados

Notebook responsável por ler os dados brutos de `data/raw/`, tratar
valores faltantes/inconsistentes, padronizar formatos e datas, e salvar
o resultado limpo em `data/processed/`.

In [1]:
import pandas as pd

df = pd.read_csv("../data/raw/lavras_clima_2000_2024.csv")
df.shape

(9132, 4)

## Valores faltando e duplicatas

Antes de qualquer tratamento, checamos se existem valores nulos (`NaN`)
por coluna e se há linhas duplicadas (mesmo dia registrado mais de uma
vez).

In [2]:
print("Valores faltando por coluna:")
print(df.isna().sum())
print("\nLinhas duplicadas:", df.duplicated().sum())

Valores faltando por coluna:
time                  0
temperature_2m_max    0
temperature_2m_min    0
precipitation_sum     0
dtype: int64

Linhas duplicadas: 0


## Datas: converter e checar dias faltando

A coluna `time` vem como texto (string). Convertendo para datetime
conseguimos usar operações de data e, mais importante, checar se o
calendário está completo (25 anos de dados diários não devem ter
"buracos").

In [3]:
df["time"] = pd.to_datetime(df["time"])

calendario_completo = pd.date_range(start=df["time"].min(), end=df["time"].max(), freq="D")
dias_faltando = calendario_completo.difference(df["time"])

print("Dias esperados:", len(calendario_completo))
print("Dias faltando:", len(dias_faltando))

Dias esperados: 9132
Dias faltando: 0


## Consistência lógica

Checagens de bom senso: a temperatura máxima do dia nunca deve ser
menor que a mínima, e a precipitação não pode ser negativa.

In [4]:
max_menor_que_min = (df["temperature_2m_max"] < df["temperature_2m_min"]).sum()
chuva_negativa = (df["precipitation_sum"] < 0).sum()

print("Linhas com max < min:", max_menor_que_min)
print("Linhas com chuva negativa:", chuva_negativa)

Linhas com max < min: 0
Linhas com chuva negativa: 0


## Salvar dado limpo

Nenhum problema encontrado nas checagens acima, então salvamos o
DataFrame (já com a coluna `time` convertida para datetime) em
`data/processed/lavras_clima_limpo.csv`, pronto para a análise
exploratória.

In [5]:
caminho_saida = "../data/processed/lavras_clima_limpo.csv"
df.to_csv(caminho_saida, index=False)
print(f"Arquivo salvo em {caminho_saida}")

Arquivo salvo em ../data/processed/lavras_clima_limpo.csv


# Produção cafeeira (IBGE/SIDRA)

Mesmo processo de limpeza aplicado à série anual de produção de café
coletada em `01_coleta.ipynb`: checar valores faltando/duplicados e
consistência lógica antes de salvar em `data/processed/`.

In [6]:
df_cafe = pd.read_csv("../data/raw/lavras_producao_cafe_2000_2024.csv")
df_cafe.shape

(25, 4)

In [7]:
print("Valores faltando por coluna:")
print(df_cafe.isna().sum())
print("\nAnos duplicados:", df_cafe["ano"].duplicated().sum())
print("Anos cobertos:", df_cafe["ano"].min(), "a", df_cafe["ano"].max(), "-", df_cafe["ano"].nunique(), "anos")

Valores faltando por coluna:
ano                       0
quantidade_produzida_t    0
rendimento_medio_kg_ha    0
area_colhida_ha           0
dtype: int64

Anos duplicados: 0
Anos cobertos: 2000 a 2024 - 25 anos


## Consistência lógica

Nenhuma das três variáveis (área colhida, quantidade produzida,
rendimento médio) pode ser negativa, e o rendimento médio informado
deve bater com `quantidade_produzida_t * 1000 / area_colhida_ha`
(conversão de toneladas para kg/ha), já que é assim que o IBGE calcula
esse indicador.

In [8]:
valores_negativos = (df_cafe[["area_colhida_ha", "quantidade_produzida_t", "rendimento_medio_kg_ha"]] < 0).sum().sum()

rendimento_calculado = df_cafe["quantidade_produzida_t"] * 1000 / df_cafe["area_colhida_ha"]
diferenca_rendimento = (rendimento_calculado - df_cafe["rendimento_medio_kg_ha"]).abs()

print("Valores negativos:", valores_negativos)
print("Maior diferença entre rendimento informado e calculado (kg/ha):", diferenca_rendimento.max())

Valores negativos: 0
Maior diferença entre rendimento informado e calculado (kg/ha): 0.4035874439462077


## Salvar dado limpo

Nenhum problema encontrado nas checagens acima, então salvamos em
`data/processed/lavras_producao_cafe_limpo.csv`.

In [9]:
caminho_saida_cafe = "../data/processed/lavras_producao_cafe_limpo.csv"
df_cafe.to_csv(caminho_saida_cafe, index=False)
print(f"Arquivo salvo em {caminho_saida_cafe}")

Arquivo salvo em ../data/processed/lavras_producao_cafe_limpo.csv
